In [1]:
!pip install cvxpy control --quiet
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install casadi


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
"""
MHE Standalone — PARÁMETROS PLANTA REAL
========================================
Traducción fiel del script MATLAB mhe_standalone_planta_real.m

CAMBIOS respecto a versión benchmark:
  [P1] Parámetros físicos → planta real calibrada
  [P2] Referencias: 0.10→0.20→0.15 m (rango físico tanques)
  [P3] Límite físico tanques: 0.25 m en lugar de 0.75 m
  [P4] σ = 1 mm (HC-SR04 real) en lugar de 2 mm
  [P5] SS calculados para rango real {0.10, 0.15, 0.20} m

Mantiene sin cambio:
  - Estructura MHE (N=20, IPOPT via CasADi)
  - Pesos S, Q, R
  - rng(42) para reproducibilidad
  - Métricas RMSE_SS con exclusión de 60s post-cambio
"""

import numpy as np
from scipy.optimize import fsolve
import casadi as ca
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import time

output_dir = Path('./outputs_sim')
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# [P1] PARÁMETROS FÍSICOS — PLANTA REAL CALIBRADA
# ============================================================================
p = {
    'Atank'   : 3.02e-3,
    'rho'     : 997.0, 'eta': 8.9e-4, 'g': 9.81,
    'alphao1' : 0.0271,  'Do1': 10e-3,
    'alphao2' : 0.0271,  'A2' : 7.85e-5,
    'alphao3' : 0.0271,  'Do3': 10e-3,
    'alpha120': 0.3038,  'D12': 10e-3,  'A12': 7.85e-5,  'lambdac12': 24000,
    'alpha230': 0.1344,  'D23': 10e-3,  'A23': 7.85e-5,  'lambdac23': 29600,
    'qmax1'   : 2.69e-5,
    'qmax3'   : 2.41e-5,
    'Ts'      : 2.0,
}
Ts = p['Ts']; EPS = 1e-10

# ============================================================================
# MODELO NO LINEAL
# ============================================================================
def nl_ode(h, u):
    h1 = max(h[0], EPS); h2 = max(h[1], EPS); h3 = max(h[2], EPS)
    qi1 = p['qmax1'] * np.clip(u[0], 0, 1)
    qi3 = p['qmax3'] * np.clip(u[1], 0, 1)
    qo1 = p['alphao1'] * (p['Do1']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h1)
    qo2 = p['alphao2'] * p['A2']                    * np.sqrt(2 * p['g'] * h2)
    qo3 = p['alphao3'] * (p['Do3']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h3)
    dh12 = h1 - h2
    l12  = p['D12'] * p['rho'] / p['eta'] * np.sqrt(2 * p['g'] * abs(dh12) + EPS)
    a12  = p['alpha120'] * np.tanh(2 * l12 / p['lambdac12'])
    q12  = a12 * p['A12'] * np.sqrt(2 * p['g'] * abs(dh12) + EPS) * np.sign(dh12 + 1e-15)
    dh23 = h2 - h3
    l23  = p['D23'] * p['rho'] / p['eta'] * np.sqrt(2 * p['g'] * abs(dh23) + EPS)
    a23  = p['alpha230'] * np.tanh(2 * l23 / p['lambdac23'])
    q23  = a23 * p['A23'] * np.sqrt(2 * p['g'] * abs(dh23) + EPS) * np.sign(dh23 + 1e-15)
    return np.array([
        (qi1 - q12 - qo1) / p['Atank'],
        (q12 - q23 - qo2) / p['Atank'],
        (qi3 + q23 - qo3) / p['Atank'],
    ])

def rk4_step(x, u, dt=Ts):
    k1 = nl_ode(x, u); k2 = nl_ode(x + dt / 2 * k1, u)
    k3 = nl_ode(x + dt / 2 * k2, u); k4 = nl_ode(x + dt * k3, u)
    return np.clip(x + dt / 6 * (k1 + 2*k2 + 2*k3 + k4), 0, 0.25)  # [P3]

# ============================================================================
# MODELO CasADi + RK4
# ============================================================================
nx, nu, ny = 3, 2, 2; eps_s = 1e-8

x_s = ca.SX.sym('x', nx); u_s = ca.SX.sym('u', nu)
h1_s, h2_s, h3_s = x_s[0], x_s[1], x_s[2]

qi1_s = p['qmax1'] * u_s[0]; qi3_s = p['qmax3'] * u_s[1]
h1p = (h1_s + ca.sqrt(h1_s**2 + eps_s)) / 2
h2p = (h2_s + ca.sqrt(h2_s**2 + eps_s)) / 2
h3p = (h3_s + ca.sqrt(h3_s**2 + eps_s)) / 2

qo1_s = p['alphao1'] * (p['Do1']**2 * np.pi / 4) * ca.sqrt(2 * p['g'] * h1p)
qo2_s = p['alphao2'] * p['A2']                    * ca.sqrt(2 * p['g'] * h2p)
qo3_s = p['alphao3'] * (p['Do3']**2 * np.pi / 4) * ca.sqrt(2 * p['g'] * h3p)

dh12_s = h1_s - h2_s; abs12_s = ca.sqrt(dh12_s**2 + eps_s)
l12_s  = p['D12'] * p['rho'] / p['eta'] * ca.sqrt(2 * p['g'] * abs12_s)
a12_s  = p['alpha120'] * ca.tanh(2 * l12_s / p['lambdac12'])
q12_s  = a12_s * p['A12'] * ca.sqrt(2 * p['g'] * abs12_s) * ca.sign(dh12_s)

dh23_s = h2_s - h3_s; abs23_s = ca.sqrt(dh23_s**2 + eps_s)
l23_s  = p['D23'] * p['rho'] / p['eta'] * ca.sqrt(2 * p['g'] * abs23_s)
a23_s  = p['alpha230'] * ca.tanh(2 * l23_s / p['lambdac23'])
q23_s  = a23_s * p['A23'] * ca.sqrt(2 * p['g'] * abs23_s) * ca.sign(dh23_s)

xdot_s = ca.vertcat(
    (qi1_s - q12_s - qo1_s) / p['Atank'],
    (q12_s - q23_s - qo2_s) / p['Atank'],
    (qi3_s + q23_s - qo3_s) / p['Atank'],
)

f_cas = ca.Function('f', [x_s, u_s], [xdot_s])
k1r = f_cas(x_s, u_s); k2r = f_cas(x_s + Ts/2*k1r, u_s)
k3r = f_cas(x_s + Ts/2*k2r, u_s); k4r = f_cas(x_s + Ts*k3r, u_s)
F_rk4 = ca.Function('F', [x_s, u_s], [x_s + Ts/6*(k1r + 2*k2r + 2*k3r + k4r)])

C_mat = np.array([[1, 0, 0], [0, 0, 1]])
print('Modelo RK4 listo (parámetros planta real).')

# ============================================================================
# MHE SETUP
# ============================================================================
print('Construyendo MHE...')
N_mhe = 20
S_mhe = np.diag([50.0, 0.5, 50.0])
Q_mhe = np.diag([10.0, 50.0, 10.0])
R_mhe = np.diag([1.0, 1.0])

xmin_mhe = np.array([0.0, 0.0, 0.0])
xmax_mhe = np.array([0.25, 0.25, 0.25])  # [P3]

X_mhe_s  = ca.SX.sym('X', nx, N_mhe + 1)
xbar_s   = ca.SX.sym('xbar', nx)
Uk_s     = ca.SX.sym('Uk', nu, N_mhe)
Yk_s     = ca.SX.sym('Yk', ny, N_mhe)

J_mhe = 0
e0 = X_mhe_s[:, 0] - xbar_s
J_mhe += ca.mtimes(e0.T, ca.DM(S_mhe) @ e0)

for j in range(N_mhe):
    xj = X_mhe_s[:, j]; xj1 = X_mhe_s[:, j + 1]
    uj = Uk_s[:, j]; yj = Yk_s[:, j]
    wj = xj1 - F_rk4(xj, uj); vj = yj - ca.DM(C_mat) @ xj
    J_mhe += ca.mtimes(wj.T, ca.DM(Q_mhe) @ wj) + ca.mtimes(vj.T, ca.DM(R_mhe) @ vj)

w_mhe   = ca.reshape(X_mhe_s, nx * (N_mhe + 1), 1)
par_mhe = ca.vertcat(xbar_s,
                     ca.reshape(Uk_s, nu * N_mhe, 1),
                     ca.reshape(Yk_s, ny * N_mhe, 1))
lbw_mhe = np.tile(xmin_mhe, N_mhe + 1)
ubw_mhe = np.tile(xmax_mhe, N_mhe + 1)

nlp_mhe = {'x': w_mhe, 'f': J_mhe, 'p': par_mhe}
opts_mhe = {'print_time': 0, 'ipopt.print_level': 0,
             'ipopt.max_iter': 200, 'ipopt.tol': 1e-4}
solver_mhe = ca.nlpsol('solver_mhe', 'ipopt', nlp_mhe, opts_mhe)
print(f'MHE listo (N={N_mhe}, IPOPT).')

# ============================================================================
# ESTADOS ESTACIONARIOS
# ============================================================================
def compute_ss(h2_ref):
    def eqs(v):
        h1, h3, u1 = v
        return nl_ode([max(h1, 1e-4), h2_ref, max(h3, 1e-4)],
                      [np.clip(u1, 0, 1), 0.0])
    guesses = [
        (h2_ref*2.0, h2_ref*0.8, 0.6), (h2_ref*1.4, h2_ref*0.8, 0.5),
        (h2_ref*1.6, h2_ref*0.7, 0.7), (h2_ref*2.5, h2_ref*0.8, 0.7),
    ]
    best, best_res = None, 1e10
    for g in guesses:
        try:
            sol = fsolve(eqs, list(g), full_output=True)
            res_norm = np.linalg.norm(sol[1]['fvec'])
            if res_norm < best_res: best_res = res_norm; best = sol[0]
        except: pass
    if best is not None and best_res < 1e-6:
        return np.array([best[0], h2_ref, best[1]]), np.array([np.clip(best[2], 0, 1), 0.0])
    return None, None

print('\nCalculando estados estacionarios...')
SS = {}
for h2r in [0.10, 0.15, 0.20]:
    x_ss, u_ss = compute_ss(h2r)
    if x_ss is not None:
        SS[h2r] = (x_ss, u_ss)
        f_chk = nl_ode(x_ss, u_ss)
        print(f'  h2={h2r:.2f}m → h=[{x_ss[0]:.4f} {x_ss[1]:.4f} {x_ss[2]:.4f}]'
              f'  u1={u_ss[0]:.4f}  |f|={np.linalg.norm(f_chk):.2e}')
    else:
        print(f'  h2={h2r:.2f}m → fsolve falló')

def get_u_ss(yr):
    for ref, (_, u) in SS.items():
        if abs(ref - yr) < 1e-3: return u.copy()
    return SS[min(SS.keys(), key=lambda k: abs(k - yr))][1].copy()

# ============================================================================
# CONFIGURACIÓN SIMULACIÓN
# ============================================================================
T_sim  = 800; Nsim = int(T_sim / Ts)
t1 = int(Nsim * 0.33); t2 = int(Nsim * 0.66)
yref_v = np.concatenate([
    0.10 * np.ones(t1), 0.20 * np.ones(t2 - t1), 0.15 * np.ones(Nsim - t2),
])
ref_change_times = [t1 * Ts, t2 * Ts]
SETTLING_S   = 60
sigma_noise  = 1e-3  # [P4]

X_true = np.zeros((nx, Nsim + 1)); X_est = np.zeros((nx, Nsim + 1))
U_hist = np.zeros((nu, Nsim)); Y_hist = np.zeros((ny, Nsim))
T_hist = np.arange(Nsim + 1) * Ts
step_ms = []

x_ss0, u_ss0 = SS[0.10]
X_true[:, 0] = x_ss0 + np.array([0.005, -0.008, 0.004])
X_est[:, 0]  = x_ss0.copy()

Uk_buf = np.tile(u_ss0.reshape(-1, 1), (1, N_mhe))
Yk_buf = np.tile((C_mat @ x_ss0).reshape(-1, 1), (1, N_mhe))
xbar   = x_ss0.copy()
w0_mhe = np.tile(x_ss0, N_mhe + 1)

rng = np.random.default_rng(42)

print(f'\nSimulando {Nsim} pasos | 0.10→0.20→0.15m | σ=1mm...')
tic = time.time()

# ============================================================================
# BUCLE PRINCIPAL
# ============================================================================
for k in range(Nsim):
    yr = yref_v[k]; uk = get_u_ss(yr); U_hist[:, k] = uk
    t0 = time.time()

    y_noisy = C_mat @ X_true[:, k] + sigma_noise * rng.standard_normal(ny)
    y_noisy = np.maximum(y_noisy, 0); Y_hist[:, k] = y_noisy

    Uk_buf = np.hstack([Uk_buf[:, 1:], uk.reshape(-1, 1)])
    Yk_buf = np.hstack([Yk_buf[:, 1:], y_noisy.reshape(-1, 1)])

    par_mhe_k = np.concatenate([xbar,
                                 Uk_buf.flatten(order='F'),
                                 Yk_buf.flatten(order='F')])
    sol_mhe   = solver_mhe(x0=w0_mhe, lbx=lbw_mhe, ubx=ubw_mhe, p=par_mhe_k)
    X_opt_mhe = np.array(sol_mhe['x']).reshape(N_mhe + 1, nx).T
    x_hat     = X_opt_mhe[:, -1]
    xbar      = X_opt_mhe[:, 1]
    w0_mhe    = np.concatenate([X_opt_mhe[:, 1:].flatten(order='F'), X_opt_mhe[:, -1]])
    X_est[:, k] = x_hat
    step_ms.append((time.time() - t0) * 1000)

    X_true[:, k + 1] = np.clip(rk4_step(X_true[:, k], uk), 0, 0.25)

    if k % 50 == 0:
        print(f'  t={k*Ts:4.0f}s | h2_true={X_true[1,k+1]:.3f}'
              f' | h2_est={x_hat[1]:.3f} | ref={yr:.3f} | {step_ms[-1]:.0f}ms')

X_est[:, -1] = X_est[:, -2]
print(f'Completado en {time.time()-tic:.1f}s')

# ============================================================================
# MÉTRICAS
# ============================================================================
idx_400 = T_hist[:Nsim] > 400
settling_mask = np.ones(Nsim, dtype=bool)
for tc in ref_change_times:
    settling_mask &= ~((T_hist[:Nsim] >= tc) & (T_hist[:Nsim] < tc + SETTLING_S))
mask_ss = idx_400 & settling_mask

e1 = (X_true[0, :Nsim] - X_est[0, :Nsim]) * 100
e2 = (X_true[1, :Nsim] - X_est[1, :Nsim]) * 100
e3 = (X_true[2, :Nsim] - X_est[2, :Nsim]) * 100

rmse_h1_ss = np.sqrt(np.mean(e1[mask_ss]**2))
rmse_h2_ss = np.sqrt(np.mean(e2[mask_ss]**2))
rmse_h3_ss = np.sqrt(np.mean(e3[mask_ss]**2))
avg_ms = np.mean(step_ms); p99_ms = np.percentile(step_ms, 99)

print(f'\n=== RMSE MHE Standalone — Simulación Planta Real (RMSE_SS, excl. {SETTLING_S}s) ===')
print(f'  h1 = {rmse_h1_ss:.4f} cm')
print(f'  h2 = {rmse_h2_ss:.4f} cm  (NO medido) ← variable de interés')
print(f'  h3 = {rmse_h3_ss:.4f} cm')
print(f"  mhe_rmse = {{'h1': {rmse_h1_ss:.4f}, 'h2': {rmse_h2_ss:.4f}, 'h3': {rmse_h3_ss:.4f}}}")

koopman_h1 = 0.0322; koopman_h2 = 0.0273; koopman_h3 = 0.0337  # de koopman_observer.ipynb

# ============================================================================
# GRÁFICAS — estilo unificado con Koopman Closed-Loop v3
# ============================================================================
ref_full_Nsim = np.append(yref_v, yref_v[-1])

fig, axes = plt.subplots(3, 2, figsize=(13, 11))
fig.suptitle(
    f'MHE Standalone — Simulación\n'
    f'RMSE_SS: h₁={rmse_h1_ss:.3f} h₂={rmse_h2_ss:.3f} h₃={rmse_h3_ss:.3f} cm | '
    f'media={avg_ms:.0f} ms | p99={p99_ms:.0f} ms',
    fontsize=9, fontweight='bold')

# ── Panel 1: h1 ─────────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(T_hist, X_true[0]*100, 'b', lw=1.5, label='$h_1$ real')
ax.plot(T_hist, X_est[0]*100,  'r--', lw=1.5, label='$h_1$ estimado (MHE)')
ax.set_ylabel('$h_1$ [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Tanque 1'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 2: h2 con referencia ──────────────────────────────────────────────
ax = axes[0, 1]
ax.plot(T_hist, X_true[1]*100, 'b', lw=2, label='$h_2$ real')
ax.plot(T_hist, X_est[1]*100,  'r--', lw=2, label='$h_2$ estimado (MHE)')
ax.stairs(ref_full_Nsim[:-1]*100, T_hist, color='k', linestyle='--', lw=1.5, label='Referencia')
ax.set_ylabel('$h_2$ [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Tanque 2 — NO medido'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 3: h3 ─────────────────────────────────────────────────────────────
ax = axes[1, 0]
ax.plot(T_hist, X_true[2]*100, 'b', lw=1.5, label='$h_3$ real')
ax.plot(T_hist, X_est[2]*100,  'r--', lw=1.5, label='$h_3$ estimado (MHE)')
ax.set_ylabel('$h_3$ [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Tanque 3'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 4: error de estimación ────────────────────────────────────────────
ax = axes[1, 1]
ax.plot(T_hist[:Nsim], e1, 'b', lw=1.2, label=f'$e_{{h1}}$ RMSE_SS={rmse_h1_ss:.3f} cm')
ax.plot(T_hist[:Nsim], e2, 'r', lw=1.5, label=f'$e_{{h2}}$ RMSE_SS={rmse_h2_ss:.3f} cm (NO medido)')
ax.plot(T_hist[:Nsim], e3, color=[0, .6, 0], lw=1.2, label=f'$e_{{h3}}$ RMSE_SS={rmse_h3_ss:.3f} cm')
ax.axhline(0, color='k', ls=':')
ax.axvline(400, color='gray', lw=0.8, ls=':', label='t=400s')
for i, tc in enumerate(ref_change_times):
    ax.axvspan(tc, tc + SETTLING_S, alpha=0.12, color='orange',
               label='transitorio excluido' if i == 0 else None)
ax.set_ylabel('Error [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Error de estimación')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 5: tiempo de cómputo ──────────────────────────────────────────────
ax = axes[2, 0]
ax.plot(step_ms, 'k', lw=0.8, alpha=0.7)
ax.axhline(avg_ms, color='b', lw=1.5, ls='--', label=f'Media={avg_ms:.1f} ms')
ax.axhline(p99_ms, color='r', lw=1.5, ls=':',  label=f'P99={p99_ms:.1f} ms')
ax.set_ylabel('Tiempo MHE [ms]'); ax.set_xlabel('Paso k')
ax.set_title('Tiempo de cómputo por paso')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 6: barras comparativas ────────────────────────────────────────────
ax = axes[2, 1]
xp = np.arange(3); w_bar = 0.35
ax.bar(xp - w_bar/2, [rmse_h1_ss, rmse_h2_ss, rmse_h3_ss],
       w_bar, color='steelblue', alpha=0.85, label='MHE standalone')
if all(v > 0 for v in [koopman_h1, koopman_h2, koopman_h3]):
    ax.bar(xp + w_bar/2, [koopman_h1, koopman_h2, koopman_h3],
           w_bar, color='forestgreen', alpha=0.85, label='Koopman Observer')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.6, 'Actualizar valores Koopman\ntras correr koopman_observer',
            ha='center', va='center', transform=ax.transAxes, fontsize=9, color='red')
ax.set_xticks(xp)
ax.set_xticklabels(['$h_1$', '$h_2$ (NO medida)', '$h_3$'])
ax.set_ylabel('RMSE_SS [cm]')
ax.set_title('Comparativa RMSE_SS')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
fname = output_dir / 'mhe_standalone_planta_real.png'
plt.savefig(fname, dpi=200, bbox_inches='tight'); plt.close()
print(f'\nFigura guardada: {fname}')

Modelo RK4 listo (parámetros planta real).
Construyendo MHE...
MHE listo (N=20, IPOPT).

Calculando estados estacionarios...
  h2=0.10m → h=[0.1140 0.1000 0.0816]  u1=0.3293  |f|=9.63e-16
  h2=0.15m → h=[0.1676 0.1500 0.1267]  u1=0.4039  |f|=4.07e-16
  h2=0.20m → h=[0.2208 0.2000 0.1725]  u1=0.4669  |f|=1.26e-17

Simulando 400 pasos | 0.10→0.20→0.15m | σ=1mm...
  t=   0s | h2_true=0.095 | h2_est=0.101 | ref=0.100 | 5ms
  t= 100s | h2_true=0.100 | h2_est=0.100 | ref=0.100 | 3ms
  t= 200s | h2_true=0.100 | h2_est=0.100 | ref=0.100 | 3ms
  t= 300s | h2_true=0.115 | h2_est=0.115 | ref=0.200 | 5ms
  t= 400s | h2_true=0.143 | h2_est=0.142 | ref=0.200 | 3ms
  t= 500s | h2_true=0.161 | h2_est=0.160 | ref=0.200 | 4ms
  t= 600s | h2_true=0.160 | h2_est=0.160 | ref=0.150 | 4ms
  t= 700s | h2_true=0.157 | h2_est=0.157 | ref=0.150 | 4ms
Completado en 1.4s

=== RMSE MHE Standalone — Simulación Planta Real (RMSE_SS, excl. 60s) ===
  h1 = 0.0339 cm
  h2 = 0.0291 cm  (NO medido) ← variable de interés
 